Week 3 – PySpark for Warehouse-Level
Insights

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("WarehouseStockAnalysis").getOrCreate()

# 1. Load large stock movement data
df_movements = spark.read.option("header", "true").option("inferSchema", "true").csv("stock_movements.csv")


In [3]:

# 2. Aggregate total stock per warehouse
warehouse_stock = df_movements.groupBy("warehouse_id", "product_id").agg(
    F.sum("quantity").alias("total_stock"),
    F.first("reorder_level").alias("reorder_level"),
    F.first("overstock_threshold").alias("overstock_limit")
)


In [4]:

# 3. Identify overstocked or understocked items
final_status = warehouse_stock.withColumn(
    "stock_status",
    F.when(F.col("total_stock") < F.col("reorder_level"), "UNDERSTOCKED")
     .when(F.col("total_stock") > F.col("overstock_limit"), "OVERSTOCKED")
     .otherwise("OPTIMAL")
)


In [6]:
# 4. Output file with warehouse-level stock status
output_path = "warehouse_stock_status"
final_status.coalesce(1).write.mode("overwrite").option("header", "true").csv(output_path)

final_status.show()

+------------+----------+-----------+-------------+---------------+------------+
|warehouse_id|product_id|total_stock|reorder_level|overstock_limit|stock_status|
+------------+----------+-----------+-------------+---------------+------------+
|   WH-DEL-03|       309|          3|            5|             40|UNDERSTOCKED|
|   WH-MUM-01|       412|          7|           15|             50|UNDERSTOCKED|
|   WH-MUM-01|       101|          7|           10|             60|UNDERSTOCKED|
|   WH-BLR-02|       205|        160|           20|            150| OVERSTOCKED|
+------------+----------+-----------+-------------+---------------+------------+

